# 24 — Umpire rankings

Rank plate umpires by the share of correct ball/strike calls, per season.
Minimum 10 plate games. Ranked by the low end of the 95% interval,
so an umpire with few games can't jump to the top on luck.


In [ ]:
import sys

import pandas as pd

sys.path.insert(0, "../src")
from stats_helpers import rate_table

TABLES = "../data/tables"
REPORTS = "../reports/tables"

g = pd.read_parquet(
    f"{TABLES}/pitches_graded/all_seasons.parquet",
    columns=["season", "game_pk", "is_call_correct"],
)
g = g[g["is_call_correct"].notna()]

games = pd.read_parquet(f"{TABLES}/games/all_seasons.parquet")
print(games.columns.tolist())
print(games.shape)

In [ ]:
umps = games[["game_pk", "plate_umpire_id", "plate_umpire_name"]]
g = g.merge(umps, on="game_pk", how="left")

print("pitches without an umpire:", g["plate_umpire_id"].isna().sum())
print(g.shape)

In [ ]:
plate_games = (
    g.groupby(["season", "plate_umpire_id"])["game_pk"]
    .nunique()
    .reset_index(name="plate_games")
)

rank = rate_table(
    g, ["season", "plate_umpire_id", "plate_umpire_name"], "is_call_correct"
)
rank = rank.merge(plate_games, on=["season", "plate_umpire_id"])
rank = rank[rank["plate_games"] >= 10].copy()

rank["rank"] = (
    rank.groupby("season")["ci_low"]
    .rank(ascending=False, method="min")
    .astype(int)
)
rank = rank.sort_values(["season", "rank"])

rank = rank.rename(
    columns={
        "plate_umpire_id": "official_id",
        "plate_umpire_name": "official_name",
        "trials": "called_pitches",
        "successes": "correct",
        "rate": "accuracy",
    }
)
rank = rank[
    [
        "season",
        "rank",
        "official_id",
        "official_name",
        "plate_games",
        "called_pitches",
        "correct",
        "accuracy",
        "ci_low",
        "ci_high",
    ]
]

print(rank.groupby("season").size())
rank[rank["season"] == 2026].head(15).round(4)

In [ ]:
import json

import requests

UMPIRE_ID = 596740  # Jansen Visconti
SEASON = 2026
RAW = "../data/raw/feed_live"

url = (
    "https://statsapi.mlb.com/api/v1/schedule"
    f"?sportId=1&season={SEASON}&gameType=R&hydrate=officials"
)
sched = requests.get(url, timeout=60).json()

hp_games = set()
for date in sched["dates"]:
    for game in date["games"]:
        if game["status"]["detailedState"] not in ("Final", "Completed Early"):
            continue
        for official in game.get("officials", []):
            if (
                official["officialType"] == "Home Plate"
                and official["official"]["id"] == UMPIRE_ID
            ):
                hp_games.add(game["gamePk"])

called = 0
for game_pk in sorted(hp_games):
    with open(f"{RAW}/{game_pk}.json") as fh:
        feed = json.load(fh)
    for play in feed["liveData"]["plays"]["allPlays"]:
        for event in play["playEvents"]:
            code = event.get("details", {}).get("call", {}).get("code")
            if event.get("isPitch") and code in ("B", "C"):
                called += 1

ours = rank[(rank["season"] == SEASON) & (rank["official_id"] == UMPIRE_ID)]
print("API plate games:  ", len(hp_games))
print("raw called pitches:", called)
print(ours[["official_name", "plate_games", "called_pitches"]].to_string(index=False))

In [ ]:
print(rank.groupby("season")["official_id"].nunique())
print()
print(rank.groupby("season").head(5).round(4).to_string(index=False))

rank.to_csv(f"{REPORTS}/umpire_rankings.csv", index=False)
print("\nsaved umpire_rankings.csv:", len(rank), "umpire-seasons")

In [ ]:
seasons = [2021, 2022, 2023, 2024, 2025, 2026]

wide = rank.assign(season=rank["season"].astype(int)).pivot(
    index=["official_id", "official_name"], columns="season", values="accuracy"
)
wide["seasons"] = wide[seasons].notna().sum(axis=1)
wide["change_2025_2026"] = wide[2026] - wide[2025]
wide["change_2021_2026"] = wide[2026] - wide[2021]
wide = wide.sort_values(2026, ascending=False)

print("umpires with all six seasons:", (wide["seasons"] == 6).sum())
print(wide.round(4).head(20).to_string())

wide.to_csv(f"{REPORTS}/umpire_accuracy_by_season_wide.csv")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

FIGURES = "../reports/figures"
Path(FIGURES).mkdir(parents=True, exist_ok=True)

full = wide[wide["seasons"] == 6]
league = g.groupby("season")["is_call_correct"].mean()

fig, ax = plt.subplots(figsize=(8, 5))
for _, row in full.iterrows():
    ax.plot(seasons, row[seasons] * 100, color="lightgrey", linewidth=1)
for (_, name), row in full.head(3).iterrows():
    ax.plot(seasons, row[seasons] * 100, marker="o", linewidth=2, label=name)
ax.plot(
    seasons, league.loc[seasons] * 100, color="black", linewidth=2.5,
    label="League",
)
ax.axvline(2025.5, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("Season")
ax.set_ylabel("Original call correct (%)")
ax.set_title("Each umpire's accuracy, 2021–2026 (10+ plate games per season)")
ax.legend()
fig.tight_layout()
fig.savefig(f"{FIGURES}/umpire_accuracy_by_season.png", dpi=150)
plt.show()


In [ ]:
wide["change_2021_2025"] = wide[2025] - wide[2021]

both = wide.dropna(subset=[2025, 2026])
long = wide.dropna(subset=[2021, 2026])
pre = wide.dropna(subset=[2021, 2025])

for label, frame, col in [
    ("2021 → 2025", pre, "change_2021_2025"),
    ("2025 → 2026", both, "change_2025_2026"),
    ("2021 → 2026", long, "change_2021_2026"),
]:
    up = (frame[col] > 0).sum()
    print(
        f"{label}: {up} of {len(frame)} umpires improved "
        f"| median {frame[col].median() * 100:+.2f} pp "
        f"| worst {frame[col].min() * 100:+.2f} pp "
        f"| best {frame[col].max() * 100:+.2f} pp"
    )

## Takeaways

1. **83–90 umpires qualify each season** (10+ plate games).
2. **The best umpires sit around 94–95% correct**; the league is 92–93%.
   Each umpire's interval is about ±0.7 pp, so the top 10–15 overlap —
   present them as a tier, not a strict order.
3. **The same names keep coming back**: Jansen Visconti (top 5 in 2021,
   2023, 2026), Tripp Gibson, Pat Hoberg (2nd in 2022 and 2023), Alex Tosi.
   Repeating across years means the ranking picks up skill, not luck.
4. **Ranked by the low end of the interval**, so a few good games can't
   carry an umpire to the top (Dillon Wilson 2026: 94.7% but 11 games → 4th).
5. **Checked against the Stats API**: Visconti 2026 — 28 plate games (exact),
   4,001 raw called pitches vs 3,997 graded (4 had no tracking data).
6. 2026 is graded with the ABS zone and midpoint location, 2021–2025 with
   operator zones at the front of the plate. Compare umpires *within* a
   season; across seasons, say so.
7. **Most umpires improved, not just a few.** Over the four pre-ABS seasons
   (2021 → 2025) 49 of 66 umpires improved, median +0.38 pp. In the one ABS
   season (2025 → 2026) 64 of 80 improved, median +0.49 pp — one year beat
   the previous four. Part of that may be the 2026 measurement change.
8. **Five of the 2026 top 15 are first-year umpires** (Wilson, Krupa,
   T. Jones, Jaschinski, Kelley) — small samples, but worth a look.

Output: `reports/tables/umpire_rankings.csv` — all seasons, all qualifying umpires.